# z628 - Ensamble ponderado
Mismo trio que `z623`, pero con pesos distintos en vez de promedio simple. `LR01` es el mejor individual (0.231) -- se le da mas peso. Pesos ajustables en `PARAM['pesos']`.

In [1]:
import os
import pandas as pd

In [2]:
PARAM = {
    'experimento': 'ENS04_PONDERADO',
    'kaggle_competition': 'labo-iii-2026-ba',
    'submits_a_promediar': {
        'regresion_lineal': '/home/ds/exp/LR01/linreg.csv',
        'autogluon': '/home/ds/exp/AutoGluon-01/AutoGluon_RMSE.csv',
        'lightgbm': '/home/ds/exp/LGB07_WF/LGB07_WF_submit.csv',
    },
    'pesos': {
        'regresion_lineal': 0.5,
        'autogluon': 0.25,
        'lightgbm': 0.25,
    }
}

assert abs(sum(PARAM['pesos'].values()) - 1.0) < 1e-6, "los pesos deben sumar 1"

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/ENS04_PONDERADO


## 1. Verificar que los 3 archivos existan

In [3]:
for nombre, p in PARAM['submits_a_promediar'].items():
    print(nombre, p, os.path.isfile(p))

regresion_lineal /home/ds/exp/LR01/linreg.csv True
autogluon /home/ds/exp/AutoGluon-01/AutoGluon_RMSE.csv True
lightgbm /home/ds/exp/LGB07_WF/LGB07_WF_submit.csv True


## 2. Cargar y promediar PONDERADO

In [4]:
tablas = {nombre: pd.read_csv(p) for nombre, p in PARAM['submits_a_promediar'].items()}

base = list(tablas.values())[0][["product_id"]].copy()
for nombre, t in tablas.items():
    base = base.merge(t.rename(columns={"tn": f"tn_{nombre}"}), on="product_id", how="left")

base["tn"] = sum(base[f"tn_{nombre}"] * peso for nombre, peso in PARAM['pesos'].items())

submit = base[["product_id", "tn"]]
print(submit.shape)
submit.head()

(780, 2)


,product_id,tn
0,20001,1167.513050
1,20002,1097.053528
2,20003,755.642473
3,20004,573.514844
4,20005,520.654717


## 3. Guardar y submit

In [5]:
archivo_submit = os.path.join(ruta, f"{PARAM['experimento']}_submit.csv")
submit.to_csv(archivo_submit, index=False)
print(archivo_submit)

/home/ds/exp/ENS04_PONDERADO/ENS04_PONDERADO_submit.csv


In [6]:
def kaggle_submit(competencia, archivo, mensaje):
    comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
    os.system(comando)

pesos_str = ", ".join(f"{k}={v}" for k, v in PARAM['pesos'].items())
kaggle_submit(PARAM['kaggle_competition'], archivo_submit, f"{PARAM['experimento']} ({pesos_str})")

100%|██████████| 18.7k/18.7k [00:00<00:00, 58.6kB/s]


93 submissions remaining today.
Successfully submitted to Labo III, 2026 BA